In [1]:
faqs = """What is the course fee?
The course fee is 799 rupees per month.
What is the course duration?
The course duration is seven months.
What does the course cover?
The course covers python and machine learning.
Can I watch recorded sessions?
Yes you can watch recorded sessions."""

print(faqs)

What is the course fee?
The course fee is 799 rupees per month.
What is the course duration?
The course duration is seven months.
What does the course cover?
The course covers python and machine learning.
Can I watch recorded sessions?
Yes you can watch recorded sessions.


In [2]:
from tensorflow.keras.preprocessing.text import Tokenizer
tokenizer = Tokenizer()

tokenizer.fit_on_texts([faqs])

print("Vocabulary:")
print(tokenizer.word_index)

vocab_size = len(tokenizer.word_index )+1
print("Vocab without padding : ", len(tokenizer.word_index) )
print("Vocab with padding : ", vocab_size )


Vocabulary:
{'the': 1, 'course': 2, 'is': 3, 'what': 4, 'fee': 5, 'duration': 6, 'can': 7, 'watch': 8, 'recorded': 9, 'sessions': 10, '799': 11, 'rupees': 12, 'per': 13, 'month': 14, 'seven': 15, 'months': 16, 'does': 17, 'cover': 18, 'covers': 19, 'python': 20, 'and': 21, 'machine': 22, 'learning': 23, 'i': 24, 'yes': 25, 'you': 26}
Vocab without padding :  26
Vocab with padding :  27


In [3]:
input_sequences = []

for sentence in faqs.split("\n"):
    token_ids = tokenizer.texts_to_sequences([sentence])[0]
    for end in range(2 , len(token_ids)+1):
        sequence = token_ids[:end]
        input_sequences.append(sequence)
        
        
for sequence in input_sequences[:4]:
    words = [tokenizer.index_word[i] for i in sequence]

    print("Token IDs:", sequence)
    print("Words:", words)
    print()

print("Total training sequences:", len(input_sequences))
    

Token IDs: [4, 3]
Words: ['what', 'is']

Token IDs: [4, 3, 1]
Words: ['what', 'is', 'the']

Token IDs: [4, 3, 1, 2]
Words: ['what', 'is', 'the', 'course']

Token IDs: [4, 3, 1, 2, 5]
Words: ['what', 'is', 'the', 'course', 'fee']

Total training sequences: 39


In [6]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
max_len = max(len(sequence) for sequence in input_sequences)

padded_sequences = pad_sequences(input_sequences , maxlen = max_len , padding = 'pre')

print(max_len)
print(padded_sequences.shape)



8
(39, 8)


In [8]:
X = padded_sequences[: , :-1]
y = padded_sequences[: , -1]

sequence_length = X.shape[1]
print(X.shape)
print(y.shape)


example_index = 3

print("\nInput IDs:", X[example_index])
print("Target ID:", y[example_index])

(39, 7)
(39,)

Input IDs: [0 0 0 4 3 1 2]
Target ID: 5


In [9]:
input_words = [
    tokenizer.index_word[int(token_id)]
    for token_id in X[example_index]
    if token_id != 0
]

target_word = tokenizer.index_word[int(y[example_index])]

print("Input words:", " ".join(input_words))
print("Word to predict:", target_word)

Input words: what is the course
Word to predict: fee


In [13]:
import tensorflow as tf
from tensorflow.keras import Sequential 
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense

tf.keras.utils.set_random_seed(42)

model = Sequential([
    Input(shape = (sequence_length , ) , dtype = 'int32'),
    Embedding(input_dim = vocab_size, output_dim = 100 , mask_zero = True),
    LSTM(150),
    Dense(vocab_size, activation = 'softmax')
    ])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 7, 100)         │         2,700 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 150)            │       150,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 27)             │         4,077 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 157,377 (614.75 KB)

 Trainable params: 157,377 (614.75 KB)

 Non-trainable params: 0 (0.00 B)

In [14]:
model.compile(optimizer = 'adam' , loss = 'sparse_categorical_crossentropy' , metrics = ['accuracy'])

In [15]:
history = model.fit(X ,y , epochs = 200 , batch_size = 8 , shuffle = True , verbose =1)

Epoch 1/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - accuracy: 0.0513 - loss: 3.2950   
Epoch 2/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.2564 - loss: 3.2741
Epoch 3/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.3333 - loss: 3.2543
Epoch 4/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.3333 - loss: 3.2302
Epoch 5/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.3333 - loss: 3.1981
Epoch 6/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.3077 - loss: 3.1521
Epoch 7/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.3077 - loss: 3.0827
Epoch 8/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.3077 - loss: 2.9741
Epoch 9/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.3077 - loss: 2.8058
Epoch 10/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.3077 - loss: 2.5779
Epoch 11/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.3333 - loss: 2.3406
Epoch 12/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.3333 -

In [20]:
import numpy as np

prompt = "Can i watch"

token_ids = tokenizer.texts_to_sequences([prompt])[0]

padded_prompt = pad_sequences(
    [token_ids],
    maxlen=sequence_length,
    padding="pre",
    truncating="pre"
)

probabilities = model.predict(padded_prompt, verbose=0)[0]

print("Input IDs:", padded_prompt[0])
print("Output shape:", probabilities.shape)
print("Probability sum:", probabilities.sum())
top_ids = np.argsort(probabilities)[::-1][:5]

print("\nTop 5 predictions:")
for token_id in top_ids:
    word = tokenizer.index_word.get(int(token_id), "<PAD>")
    print(f"{word}: {probabilities[token_id]:.2%}")

Input IDs: [ 0  0  0  0  7 24  8]
Output shape: (27,)
Probability sum: 1.0

Top 5 predictions:
recorded: 99.79%
watch: 0.11%
sessions: 0.08%
can: 0.01%
course: 0.00%
